In [6]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent:
    if (project_root / 'src').exists():
        break
    project_root = project_root.parent
else:
    raise RuntimeError('Unable to locate project root containing src/')

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


# RPFR Walkthrough (HD Example)

Illustrative calculation of a reduced partition function ratio (RPFR) for the H$_2$ / HD pair using the published constants from Richet et al. (1977).

In [7]:
from richet_rpfr import (
    load_molecular_constants_from_excel,
    PartitionFunctionCalculator,
    compute_and_finalize_with_temp,
)
import pandas as pd

raw_excel = project_root / 'data' / 'processed' / 'spreadsheets' / 'Richet - RPFR & mol constans - diatoms.xlsx'
constants = load_molecular_constants_from_excel(raw_excel, sheet_name='Diatoms (Table 5)')

NSI_H2 = constants['1H1H']
SSI_HD = constants['1H2H']
T_0C = 273.15

calc = PartitionFunctionCalculator(temperature=T_0C, NSI=NSI_H2, SSI=SSI_HD)
calc.calculate_all()
calc.print_contribution_table()


Contribution                  | Value
------------------------------+------
Translational                 | 1.835706e+00
ZPE (G0)                      | 1.000000e+00
ZPE (harmonic)                | 4.715118e+00
ZPE (anharmonic)              | 9.607942e-01
ZPE (total)                   | 4.530258e+00
Excited (harmonic)            | 1.000000e+00
Excited (anharmonic)          | 1.000000e+00
Rotational (linear)           | 2.656667e+00
Rotational (linear, corr.)    | 2.587940e+00
Rotational (nonlinear, corr.) | 1.000000e+00
Rotational (diatomic)         | 2.587775e+00
Rotational-vibrational        | 1.000000e+00
Q total                       | 2.152051e+01
RPFR                          | 2.152051e+01


In [8]:
paper_HD_H2 = {
    'Translational': 1.83561,
    'ZPE_G0': 1.00910,
    'ZPE_harmonic': 4.71546,
    'ZPE_anharmonic': 0.96079,
    'ZPE_total': 4.57179,
    'Rotational Diatomic': 2.59246,
    'Total': 21.75600,
}

final = compute_and_finalize_with_temp(T_0C, NSI_H2, SSI_HD, paper_HD_H2)

pd.Series(final, name='HD / H2 RPFR contributions')


Translational                1.835706
ZPE_G0                       1.009100
ZPE_harmonic                 4.715118
ZPE_anharmonic               0.960794
ZPE_total                    4.571484
Excited state harmonic       1.000000
Excited state anharmonic     1.000000
Rotational linear            2.587940
Rotational Diatomic          2.587775
Rotational-vibrational       1.000000
Total                       21.716349
Name: HD / H2 RPFR contributions, dtype: float64